# 📖 Notebook 4: Trip Lifecycle Management

A ride goes through many states: requested → matching → assigned → accepted → en_route → arrived → in_progress → completed. At each step, multiple things can go wrong: a driver might not respond, two services might try to assign the same driver, or a rider might cancel mid-match.

This notebook covers:
- The **ride state machine** — valid transitions between states
- **Distributed locking with Redis** — preventing two rides from claiming the same driver
- **TTL-based locks** — automatically releasing locks if a driver doesn't respond
- **The complete ride flow** from request to completion

## Learning Objectives

By the end of this notebook, you'll understand:
- How to model a ride as a state machine with strict transition rules
- Why distributed locks are needed (and what breaks without them)
- How Redis SET NX EX implements a lock with automatic expiry
- How to handle driver timeouts and move to the next candidate

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/uber
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import random
import json

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "uber_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

# The matching flow below rolls a die for driver accept/decline. Seed it so a
# re-run tells the same story and the assertions are not coin flips.
random.seed(42)

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

try:
    conn = get_db(); conn.close()
    print("✅ Connected to PostgreSQL + PostGIS")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis(); r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🔄 The Ride State Machine

A ride is not just "active" or "done" — it passes through many stages. Each stage has rules about which transitions are valid.

```
                     ┌─────────────┐
                     │  requested   │  Rider tapped "Request Ride"
                     └──────┬───────┘
                            │
                            ▼
                     ┌─────────────┐
            ┌────────│  matching    │  System searching for a driver
            │        └──────┬───────┘
            │               │
            │               ▼
            │        ┌──────────────────┐
            │        │ driver_assigned   │  Found a driver, waiting for response
            │        └────┬────────┬────┘
            │             │        │
            │        accept     decline/timeout
            │             │        │
            │             ▼        └──→ back to "matching"
            │        ┌─────────────┐
            │        │  accepted    │  Driver accepted the ride
            │        └──────┬───────┘
            │               │
            │               ▼
            │        ┌─────────────┐
            │        │  en_route    │  Driver heading to pickup
            │        └──────┬───────┘
            │               │
            │               ▼
            │        ┌─────────────┐
            │        │  arrived     │  Driver at pickup location
            │        └──────┬───────┘
            │               │
            │               ▼
            │        ┌─────────────┐
            │        │ in_progress  │  Rider in car, heading to destination
            │        └──────┬───────┘
            │               │
            │               ▼
            │        ┌─────────────┐
            │        │  completed   │  Arrived at destination
            │        └─────────────┘
            │
            └──────→ ┌─────────────┐
                     │  cancelled   │  (can happen from most states)
                     └─────────────┘
```

In [ ]:
# Define valid state transitions

VALID_TRANSITIONS = {
    "requested":       ["matching", "cancelled"],
    "matching":        ["driver_assigned", "cancelled"],
    "driver_assigned": ["accepted", "matching", "cancelled"],  # matching = declined/timeout
    "accepted":        ["en_route", "cancelled"],
    "en_route":        ["arrived", "cancelled"],
    "arrived":         ["in_progress", "cancelled"],
    "in_progress":     ["completed"],  # can't cancel mid-ride
    "completed":       [],  # terminal state
    "cancelled":       [],  # terminal state
}

def can_transition(current_status, new_status):
    """Check if a state transition is valid."""
    return new_status in VALID_TRANSITIONS.get(current_status, [])


def update_ride_status(ride_id, new_status):
    """
    Transition a ride to a new status.
    Validates the transition and updates the appropriate timestamp.
    """
    conn = get_db()
    conn.autocommit = True
    cur = conn.cursor()
    
    # Get current status
    cur.execute("SELECT status FROM rides WHERE id = %s;", (ride_id,))
    row = cur.fetchone()
    if not row:
        conn.close()
        return False, f"Ride {ride_id} not found"
    
    current = row[0]
    
    # Validate transition
    if not can_transition(current, new_status):
        conn.close()
        return False, f"Invalid: {current} → {new_status}"
    
    # Update status and the appropriate timestamp
    timestamp_col = {
        "accepted": "accepted_at",
        "in_progress": "pickup_at",
        "completed": "dropoff_at",
        "cancelled": "cancelled_at",
    }.get(new_status)
    
    if timestamp_col:
        cur.execute(f"""
            UPDATE rides SET status = %s, {timestamp_col} = NOW()
            WHERE id = %s;
        """, (new_status, ride_id))
    else:
        cur.execute("""
            UPDATE rides SET status = %s WHERE id = %s;
        """, (new_status, ride_id))
    
    conn.close()
    return True, f"{current} → {new_status}"


# Demonstrate valid and invalid transitions
print("🔄 State Transition Rules:")
print()

tests = [
    ("requested", "matching",        True),
    ("requested", "completed",       False),  # can't skip to completed!
    ("matching", "driver_assigned",   True),
    ("driver_assigned", "matching",   True),   # driver declined → back to matching
    ("driver_assigned", "completed",  False),  # can't skip!
    ("in_progress", "cancelled",      False),  # can't cancel mid-ride
    ("completed", "requested",        False),  # terminal state
]

for current, target, expected in tests:
    result = can_transition(current, target)
    icon = "✅" if result == expected else "❌ BUG"
    valid = "VALID" if result else "BLOCKED"
    print(f"  {icon} {current:>16} → {target:<16} {valid}")

# Printing "❌ BUG" is not the same as failing. Make it fail.
mismatches = [(c, t, can_transition(c, t), e) for c, t, e in tests if can_transition(c, t) != e]
assert not mismatches, (
    f"the state machine disagrees with the documented rules "
    f"(current, target, got, expected): {mismatches}"
)

# And prove the validator is actually wired to the database, not just to this table.
conn = get_db(); cur = conn.cursor()
cur.execute("SELECT id FROM rides WHERE status = 'completed' ORDER BY id LIMIT 1;")
row = cur.fetchone()
conn.close()
if row:
    ok, detail = update_ride_status(row[0], "requested")
    print(f"\n  Live row check — ride #{row[0]} (completed) → requested: {detail}")
    assert not ok and detail.startswith("Invalid"), (
        f"a terminal ride must not be resurrected; got {(ok, detail)}"
    )
else:
    print("\n  (no completed ride in the table to check the validator against)")

## 💥 Without a Lock: The Race Condition (bad baseline)

Here's the naive flow two ride-request workers might follow in parallel:

1. `GEOSEARCH` → get the list of nearby drivers.
2. Pick the top one.
3. `UPDATE rides SET driver_id = X`.

If two workers do this at the same instant, **both** see the same top driver as "available" and
**both** write the same `driver_id` into different ride rows. The driver now has two pickups for
the same minute. This is the classic "check-then-act" race.

We'll reproduce it **deterministically** — no thread luck needed — by interleaving the steps by hand.

In [ ]:
# Deterministic demo: two rides, one driver, no lock → double assignment.
r = get_redis()
conn = get_db(); conn.autocommit = True; cur = conn.cursor()

# Clean slate: only driver 1 is available
r.delete("drivers:available")
r.sadd("drivers:available", "driver:1")
for k in r.keys("lock:driver:*"):
    r.delete(k)

def naive_pick_top_driver():
    """Return the first available driver — no locking."""
    avail = r.smembers("drivers:available")
    return sorted(avail)[0] if avail else None

# Step 1: Worker A looks up the top driver
pick_a = naive_pick_top_driver()
print(f"  Worker A sees top driver: {pick_a}")

# Step 2: Worker B looks up BEFORE A writes — sees the same driver
pick_b = naive_pick_top_driver()
print(f"  Worker B sees top driver: {pick_b}")

# Step 3: Both workers create rides and assign the same driver
cur.execute("""INSERT INTO rides (rider_id, status, driver_id, pickup_location, dropoff_location)
               VALUES (1, 'driver_assigned', 1,
                       ST_SetSRID(ST_MakePoint(-122.42, 37.77), 4326),
                       ST_SetSRID(ST_MakePoint(-122.41, 37.78), 4326))
               RETURNING id;""")
ride_a = cur.fetchone()[0]
cur.execute("""INSERT INTO rides (rider_id, status, driver_id, pickup_location, dropoff_location)
               VALUES (2, 'driver_assigned', 1,
                       ST_SetSRID(ST_MakePoint(-122.43, 37.76), 4326),
                       ST_SetSRID(ST_MakePoint(-122.40, 37.79), 4326))
               RETURNING id;""")
ride_b = cur.fetchone()[0]

cur.execute("SELECT id, rider_id, driver_id, status FROM rides WHERE id IN (%s, %s);",
            (ride_a, ride_b))
rows = cur.fetchall()
print("\n  💥 Resulting ride rows:")
for row in rows:
    print(f"     ride {row[0]}: rider={row[1]} driver={row[2]} status={row[3]}")

# Cleanup the broken rows so they don't pollute later cells
cur.execute("DELETE FROM rides WHERE id IN (%s, %s);", (ride_a, ride_b))
conn.close()

print("\n  Both rides got driver #1. In production this is an outage-class bug:")
print("  one car, two pickups, two angry riders. We need an atomic check-and-claim.")

# A "bad baseline" that quietly behaves itself teaches nothing. Assert the bug.
assert pick_a == pick_b == "driver:1", (
    f"both workers were supposed to see the same top driver, got {pick_a} and {pick_b}"
)
double_assigned = [row[0] for row in rows if row[2] == 1]
assert len(double_assigned) == 2, (
    f"the unlocked path was supposed to put driver #1 on TWO rides, got {double_assigned}. "
    "If it did not, this cell no longer reproduces the race it claims to."
)

## 🔒 The Double-Assignment Problem

Here's a critical race condition:

```
Time    Service Instance A              Service Instance B
─────   ──────────────────              ──────────────────
  1     Ride #100 needs a driver        Ride #200 needs a driver
  2     Query: nearest driver → #5      Query: nearest driver → #5
  3     Assign driver #5 to ride #100   Assign driver #5 to ride #200
  4     ❌ Driver #5 now has TWO rides!
```

Both services found the same "best" driver and assigned them simultaneously. This violates our core requirement: **each driver should only have one active ride at a time**.

### Solution: Distributed Locks with Redis

Before assigning a driver, we **lock** them using Redis `SET key value NX EX ttl`:
- `NX` = only set if the key does **not** exist ("set if not exists")
- `EX ttl` = auto-expire after `ttl` seconds

If the SET returns True, we got the lock. If False, someone else already locked this driver.

In [ ]:
# Redis as a distributed lock: SET NX EX is atomic — only one caller wins.
# We also give each lock a unique owner token so releasing is SAFE:
# we only delete the key if it still belongs to us. Otherwise an expired-then-
# reused lock could be deleted by its previous owner.
import uuid

def acquire_driver_lock(driver_id, ride_id, ttl_seconds=10):
    """Try to lock a driver. Returns a unique token on success, else None."""
    r = get_redis()
    token = f"ride:{ride_id}:{uuid.uuid4().hex[:8]}"
    ok = r.set(f"lock:driver:{driver_id}", token, nx=True, ex=ttl_seconds)
    return token if ok else None

# Lua script: atomic "delete only if value matches". Prevents deleting someone
# else's lock after your own expired.
_SAFE_RELEASE = """
if redis.call('GET', KEYS[1]) == ARGV[1] then
  return redis.call('DEL', KEYS[1])
else
  return 0
end
"""

def release_driver_lock(driver_id, token):
    """Release a lock only if we still own it."""
    r = get_redis()
    r.eval(_SAFE_RELEASE, 1, f"lock:driver:{driver_id}", token)

def check_driver_lock(driver_id):
    r = get_redis()
    key = f"lock:driver:{driver_id}"
    return r.get(key), r.ttl(key)


# Demo the locking mechanism
print("🔒 Distributed Lock Demo:")
r = get_redis()
for key in r.keys("lock:driver:*"):
    r.delete(key)

print("  Ride #100 tries to lock driver #5...")
token_a = acquire_driver_lock(driver_id=5, ride_id=100, ttl_seconds=10)
print(f"  → {'✅ LOCKED ('+token_a+')' if token_a else '❌ FAILED'}")

print("\n  Ride #200 tries to lock driver #5...")
token_b = acquire_driver_lock(driver_id=5, ride_id=200, ttl_seconds=10)
print(f"  → {'✅ LOCKED' if token_b else '❌ FAILED — driver already locked!'}")

owner, ttl = check_driver_lock(5)
print(f"\n  Lock status: driver #5 owned by {owner}, expires in {ttl}s")

assert token_a is not None, "the first claimant must get the lock"
assert token_b is None, "SET NX must reject the second claimant while the lock is held"
assert owner == token_a, f"lock should still be ride #100's token, got {owner!r}"
assert 0 < ttl <= 10, f"the lock must carry a TTL so it cannot leak, got {ttl}"

release_driver_lock(5, token_a)
assert check_driver_lock(5)[0] is None, "the owner must be able to release its own lock"
print("\n💡 SET NX is atomic. The token + scripted release make DEL safe too.")

In [ ]:
# TTL auto-expiry: what if the driver's phone loses signal and never responds?

print("⏱️ TTL Lock Expiry Demo:")
r = get_redis()
r.delete("lock:driver:7")

print("  Locking driver #7 with 3-second TTL...")
tok = acquire_driver_lock(driver_id=7, ride_id=300, ttl_seconds=3)
owner, ttl = check_driver_lock(7)
print(f"  Lock status: owner={owner}, TTL={ttl}s")

print("  Waiting 4 seconds (driver not responding)...")
time.sleep(4)
owner, ttl = check_driver_lock(7)
print(f"  Lock status: owner={owner}, TTL={ttl}  → lock auto-expired")

print("\n  Ride #400 tries to lock driver #7...")
tok2 = acquire_driver_lock(driver_id=7, ride_id=400, ttl_seconds=10)
print(f"  → {'✅ LOCKED' if tok2 else '❌ FAILED'}")

# 🧨 The dangerous sequel: ride #300 wakes up late and calls release.
# A naive `DEL lock:driver:7` here would delete ride #400's brand-new lock and
# hand driver #7 to a third ride. The token check turns it into a no-op.
print("\n  Ride #300 (whose lock already expired) calls release...")
release_driver_lock(7, tok)
owner_after, _ = check_driver_lock(7)
print(f"  → driver #7 is still owned by {owner_after}")

assert tok2 is not None, "the lock should have expired and become re-acquirable"
assert owner_after == tok2, (
    f"ride #300's stale release deleted ride #400's lock — owner is now {owner_after!r}. "
    "This is exactly why the release is token-checked instead of a plain DEL."
)

print("\n💡 No manual cleanup. If the driver's app crashes, the system moves on.")
release_driver_lock(7, tok2)
assert check_driver_lock(7)[0] is None, "the rightful owner must be able to release"

## 🚗 Complete Ride Flow: Request to Completion

Let's put everything together and simulate a complete ride lifecycle:

1. Rider requests a ride
2. System finds nearby drivers
3. System locks the best driver and sends the request
4. If driver declines or times out → try the next driver
5. Driver accepts → ride proceeds through remaining states

In [ ]:
# Load drivers into Redis for matching
r = get_redis()
conn = get_db()
cur = conn.cursor()

r.delete("drivers:locations", "drivers:available")
for key in r.keys("lock:driver:*"):
    r.delete(key)

cur.execute("""
    SELECT d.id, d.status,
           ST_X(dl.location::geometry) AS lng,
           ST_Y(dl.location::geometry) AS lat
    FROM drivers d
    JOIN driver_locations dl ON d.id = dl.driver_id;
""")
for row in cur.fetchall():
    did, status, lng, lat = row
    r.geoadd("drivers:locations", (lng, lat, f"driver:{did}"))
    if status == "available":
        r.sadd("drivers:available", f"driver:{did}")

conn.close()
print(f"✅ Loaded drivers: {r.scard('drivers:available')} available")

In [ ]:
def request_ride(rider_id, pickup_lng, pickup_lat, dropoff_lng, dropoff_lat):
    """Full ride request flow: create → match → lock → assign."""
    conn = get_db(); conn.autocommit = True; cur = conn.cursor()
    r = get_redis()

    cur.execute("""
        INSERT INTO rides (rider_id, status, pickup_location, dropoff_location)
        VALUES (%s, 'requested',
                ST_SetSRID(ST_MakePoint(%s, %s), 4326),
                ST_SetSRID(ST_MakePoint(%s, %s), 4326))
        RETURNING id;
    """, (rider_id, pickup_lng, pickup_lat, dropoff_lng, dropoff_lat))
    ride_id = cur.fetchone()[0]
    print(f"  📱 Ride #{ride_id} created (status: requested)")

    cur.execute("UPDATE rides SET status = 'matching' WHERE id = %s;", (ride_id,))
    print(f"  🔍 Status → matching")

    nearby = r.geosearch(name="drivers:locations",
                        longitude=pickup_lng, latitude=pickup_lat,
                        radius=5, unit="km",
                        withdist=True, sort="ASC", count=10)
    available = r.smembers("drivers:available")
    candidates = [(m, d) for m, d in nearby if m in available]
    print(f"  📋 Found {len(candidates)} nearby available drivers")

    for driver_key, distance in candidates:
        driver_id = int(driver_key.split(":")[1])
        token = acquire_driver_lock(driver_id, ride_id, ttl_seconds=10)
        if not token:
            print(f"  ❌ {driver_key} is locked (assigned to another ride)")
            continue

        # Re-check availability INSIDE the lock. `candidates` was built before we
        # held anything, so another worker may have claimed this driver in the gap.
        # Taking a lock without re-reading the state it protects just moves the
        # race one step later; it does not remove it.
        if not r.sismember("drivers:available", driver_key):
            print(f"  ↩️  {driver_key} was claimed while we waited — releasing, next")
            release_driver_lock(driver_id, token)
            continue

        print(f"  🔒 Locked {driver_key} ({float(distance):.2f} km away)")

        cur.execute("""UPDATE rides SET status = 'driver_assigned', driver_id = %s
                     WHERE id = %s;""", (driver_id, ride_id))
        print(f"  📤 Status → driver_assigned (sent request to {driver_key})")

        accepts = random.random() < 0.8
        if accepts:
            print(f"  ✅ {driver_key} ACCEPTED!")
            cur.execute("""UPDATE rides SET status = 'accepted', accepted_at = NOW()
                         WHERE id = %s;""", (ride_id,))
            # The SREM is what protects the driver from here on. The lock only
            # covers the assignment window; once the driver is off the available
            # set, nobody else can pick them as a candidate.
            r.srem("drivers:available", driver_key)
            release_driver_lock(driver_id, token)
            conn.close()
            return ride_id, driver_id
        else:
            print(f"  ❌ {driver_key} DECLINED, trying next...")
            release_driver_lock(driver_id, token)
            cur.execute("UPDATE rides SET status = 'matching', driver_id = NULL WHERE id = %s;", (ride_id,))

    cur.execute("UPDATE rides SET status = 'cancelled', cancelled_at = NOW() WHERE id = %s;", (ride_id,))
    print(f"  ❌ No drivers available — ride cancelled")
    conn.close()
    return ride_id, None


print("=" * 60)
print("🚗 Requesting a ride: Downtown SF → Mission District")
print("=" * 60)

ride_id, driver_id = request_ride(
    rider_id=1,
    pickup_lng=-122.4194, pickup_lat=37.7749,
    dropoff_lng=-122.4103, dropoff_lat=37.7627
)

assert ride_id is not None, "request_ride must always create a ride row"
if driver_id is not None:
    assert not r.sismember("drivers:available", f"driver:{driver_id}"), (
        f"driver:{driver_id} was assigned but is still in drivers:available — "
        "a second rider could be matched to the same car"
    )
    assert not r.exists(f"lock:driver:{driver_id}"), (
        f"lock on driver:{driver_id} outlived the assignment; it should be released "
        "as soon as the ride is accepted"
    )

In [ ]:
# Complete the ride lifecycle (if a driver was assigned)

if driver_id:
    print(f"\n🚗 Completing ride #{ride_id} with driver #{driver_id}")
    print("=" * 50)
    
    steps = [
        ("en_route",    "🚗 Driver heading to pickup..."),
        ("arrived",     "📍 Driver arrived at pickup!"),
        ("in_progress", "🛣️  Rider picked up, heading to destination..."),
        ("completed",   "🏁 Arrived! Ride complete."),
    ]
    
    for new_status, message in steps:
        time.sleep(0.5)  # small delay for readability
        success, detail = update_ride_status(ride_id, new_status)
        icon = "✅" if success else "❌"
        print(f"  {icon} {message} ({detail})")
    
    # Mark driver as available again
    r = get_redis()
    r.sadd("drivers:available", f"driver:{driver_id}")
    print(f"\n  🟢 Driver #{driver_id} is available again")
    
    # Show the final ride record
    conn = get_db()
    cur = conn.cursor()
    cur.execute("""
        SELECT id, rider_id, driver_id, status,
               requested_at, accepted_at, pickup_at, dropoff_at
        FROM rides WHERE id = %s;
    """, (ride_id,))
    row = cur.fetchone()
    conn.close()
    
    print(f"\n  📋 Final Ride Record:")
    print(f"     Ride ID:      {row[0]}")
    print(f"     Rider:        {row[1]}")
    print(f"     Driver:       {row[2]}")
    print(f"     Status:       {row[3]}")
    print(f"     Requested:    {row[4]}")
    print(f"     Accepted:     {row[5]}")
    print(f"     Picked up:    {row[6]}")
    print(f"     Dropped off:  {row[7]}")
else:
    print("\n❌ No driver was assigned. Try running the cell again!")

## 🏎️ Concurrent Ride Requests (Race Condition Test)

Let's prove that our locking prevents double-assignment by requesting two rides simultaneously that both want the same driver.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

r = get_redis()
conn = get_db(); conn.autocommit = True; cur = conn.cursor()

for key in r.keys("lock:driver:*"):
    r.delete(key)

# Snapshot the availability set so we can put it back exactly as we found it.
saved_available = r.smembers("drivers:available")

# Force both rides to compete for the one and only available driver.
r.delete("drivers:available")
r.sadd("drivers:available", "driver:1")
print(f"Available drivers: {r.smembers('drivers:available')}\n")


def claim_driver(rider_id, driver_id=1):
    """The same check-then-act as the naive cell — but fenced.

    Order matters: take the lock FIRST, then re-read the state it protects,
    then write. Checking before locking is how you get the race back.
    """
    own_conn = get_db(); own_conn.autocommit = True; own_cur = own_conn.cursor()
    own_cur.execute("""
        INSERT INTO rides (rider_id, status, pickup_location, dropoff_location)
        VALUES (%s, 'matching',
                ST_SetSRID(ST_MakePoint(-122.42, 37.77), 4326),
                ST_SetSRID(ST_MakePoint(-122.41, 37.78), 4326))
        RETURNING id;
    """, (rider_id,))
    ride_id = own_cur.fetchone()[0]

    def bail(reason):
        own_cur.execute(
            "UPDATE rides SET status = 'cancelled', cancelled_at = NOW() WHERE id = %s;",
            (ride_id,))
        own_conn.close()
        return ride_id, None, reason

    token = acquire_driver_lock(driver_id, ride_id, ttl_seconds=10)
    if token is None:
        return bail("lost the lock race")

    if not r.sismember("drivers:available", f"driver:{driver_id}"):
        release_driver_lock(driver_id, token)
        return bail("got the lock, but the driver was already taken")

    r.srem("drivers:available", f"driver:{driver_id}")
    own_cur.execute(
        "UPDATE rides SET status = 'driver_assigned', driver_id = %s WHERE id = %s;",
        (driver_id, ride_id))
    own_conn.close()
    return ride_id, token, "assigned"


print("🏎️ Two rides racing for driver #1, both going through the lock:\n")
with ThreadPoolExecutor(max_workers=2) as pool:
    results = [f.result() for f in [pool.submit(claim_driver, rid) for rid in (1, 2)]]

for ride_id, token, outcome in results:
    print(f"  {'✅' if token else '🛑'} ride #{ride_id}: {outcome}")

# Don't take the workers' word for it — ask the database.
ride_ids = tuple(ride_id for ride_id, _, _ in results)
cur.execute("SELECT id, driver_id, status FROM rides WHERE id IN %s ORDER BY id;", (ride_ids,))
rows = cur.fetchall()
print("\n  📋 Ride rows as stored:")
for row in rows:
    print(f"     ride {row[0]}: driver={row[1]} status={row[2]}")

winners = [(ride_id, token) for ride_id, token, _ in results if token]
holds_driver_1 = [row[0] for row in rows if row[1] == 1]

assert len(winners) == 1, f"exactly one ride should win the lock, got {winners}"
assert len(holds_driver_1) == 1, (
    f"driver #1 ended up on {len(holds_driver_1)} rides ({holds_driver_1}) — the lock "
    "did not fence the race"
)
assert r.scard("drivers:available") == 0, (
    "the winning ride should have removed driver #1 from the available set"
)

print(f"\n✅ Only ride #{holds_driver_1[0]} holds driver #1.")
print("   The unlocked version earlier in this notebook produced TWO. Same two")
print("   workers, same driver, same instant — the only difference is the lock.")

# Put the world back: drop the demo rows, release the lock, restore availability.
cur.execute("DELETE FROM rides WHERE id IN %s;", (ride_ids,))
conn.close()
for _, token in winners:
    release_driver_lock(1, token)
r.delete("drivers:available")
for member in saved_available:
    r.sadd("drivers:available", member)

## 🪪 A Quick Word on Idempotency

What if the rider taps "Request Ride" twice — or their phone retries the HTTP call because
the network dropped? Without protection, you'd create two `rides` rows for the same intent.

The standard fix is an **idempotency key**: the client generates a random UUID for each
user intent and sends it as a header. The server stores `(rider_id, idempotency_key) → ride_id`
and, on retries, returns the existing `ride_id` instead of creating a new one. This is a small
addition, but it's what keeps one tap from turning into two cars.

```python
# Sketch: server-side handling
key = request.headers["Idempotency-Key"]
existing = r.get(f"idem:{rider_id}:{key}")
if existing:
    return {"ride_id": int(existing)}     # safe retry
ride_id = create_ride(...)
r.set(f"idem:{rider_id}:{key}", ride_id, ex=24*3600)
```

## 🧹 Cleanup

In [ ]:
r = get_redis()
for key in r.keys("lock:driver:*"):
    r.delete(key)
r.delete("drivers:locations", "drivers:available")
print("🧹 Cleaned up Redis keys")

## 📚 Summary

### Key Takeaways

1. **State machines** enforce valid ride transitions — you can't skip from "requested" to "completed"
2. **Distributed locks** (Redis SET NX EX) prevent double-assignment of drivers
3. **TTL on locks** means unresponsive drivers don't block the system — the lock auto-expires
4. **The matching loop** tries drivers in distance order, skipping any that are already locked
5. **Concurrency safety** — even with multiple service instances, only one ride can claim a driver
6. **A lock is not enough on its own** — you must re-read the state the lock protects *after*
   acquiring it. Locking around a candidate list you built beforehand just moves the race
   one step later
7. **The lock only covers the assignment window** — once the driver is accepted and removed
   from `drivers:available`, that removal is the durable guard. The lock is released
   immediately so a declined driver isn't held hostage for the full TTL

### How This Fits in a System Design Interview

The ride lifecycle question tests:
- **State management** — can you model complex workflows as state machines?
- **Consistency** — how do you prevent race conditions in a distributed system?
- **Failure handling** — what happens if a driver/service crashes mid-assignment?
- **Lock strategies** — application locks vs. database locks vs. distributed locks (Redis)

The progression:
- ❌ Bad: application-level locks → no coordination between instances
- ✅ Good: database locks → coordinated but locks stuck if service crashes
- ✅ Great: Redis distributed locks with TTL → atomic, coordinated, auto-expiring

And the honest footnote: a single-node Redis lock is not a correctness guarantee. If the
Redis primary fails over before replicating the key, two holders can exist at once. Real
systems either accept that window (assignment is cheap to undo) or use a fencing token that
the database checks on write. This lab does the former.

### 🎓 What We Covered in This Lab Series

| Notebook | Concept | Key Technology |
|----------|---------|----------------|
| 1. Geospatial Matching | Finding nearby drivers | PostGIS + Redis GEOSEARCH |
| 2. Real-Time Tracking | Handling millions of location updates | Redis GEOADD + staleness cleanup |
| 3. Surge Pricing | Dynamic pricing from supply/demand | Zone counting + multiplier formula |
| 4. Trip Lifecycle | Ride state machine + preventing race conditions | State transitions + Redis distributed locks |